**Author:** Salvador Navas  
**Date:** 2025-06-27

### PERSIANN-CCS — Satellite Precipitation Estimation

**PERSIANN-CCS** (Precipitation Estimation from Remotely Sensed Information using
Artificial Neural Networks – Cloud Classification System) is produced by the
**CHRS at UC Irvine** using infrared brightness temperatures from geostationary satellites.

| Property | Value |
|----------|-------|
| Spatial resolution | ~0.04° (~4 km) |
| Temporal resolution | **1 hour** |
| Coverage | 60°S – 60°N |
| Period | 2003 – present |
| Latency | ~1 day (near real-time) |

#### 🛰️ How it works

PERSIANN-CCS estimates rainfall from **infrared (IR) cloud-top temperatures** from
geostationary satellites (GOES, Meteosat, Himawari). Cold cloud tops = deep convective towers = rain.
An artificial neural network translates IR temperature patterns to rainfall rates,
using the **Cloud Classification System (CCS)** to distinguish convective from stratiform clouds.

#### 🆚 PERSIANN-CCS vs GPM IMERG — when to use which?

| Feature | PERSIANN-CCS | GPM IMERG |
|---------|-------------|----------|
| Algorithm | IR-only + ANN | Multi-sensor (MW + IR + gauge) |
| Temporal resolution | **1 hour** | 30 min |
| Spatial resolution | ~4 km | ~10 km |
| Accuracy (tropics) | Moderate | Better |
| Accuracy (mid-latitudes, frontal rain) | Lower | Better |
| Near real-time latency | ~1 day | ~4 hours (early run) |
| Period | 2003–now | 2000–now |
| Access | Anonymous FTP | NASA Earthdata (free) |

> **Use PERSIANN-CCS** when you need **near real-time hourly** data (flash flood warning,
> operational monitoring) or the finer spatial resolution (~4 km vs ~10 km) matters.

> **Use GPM IMERG** for historical analysis, bias correction, or when accuracy is more important
> than latency — especially for frontal/orographic rainfall in Mediterranean/Atlantic Spain.

#### ⚠️ Known limitations

- **Underestimates peak intensity** of convective cells (IR signal saturates for deep convection)
- **False detections** in orographic fog and thin cirrus (cold but not precipitating)
- **Accuracy decreases** north of ~45°N (frontal systems harder to detect from IR)

#### 🔗 Reference
- Hong et al. (2004), *Journal of Applied Meteorology*, 43(12). https://doi.org/10.1175/JAM2180.1


In [1]:
import os
from pathlib import Path

from pyhydra.data_sources.rainfall import PERSSIANDownloader
from pyhydra.utils import interactive_map

# Portable repo-root / data-dir resolution (local clone, Docker, Azure ML)
_cwd = Path.cwd()
_candidates = [Path('/workspace'), _cwd, *_cwd.parents]
REPO_ROOT = next(
    (p for p in _candidates if (p / 'notebooks').exists() or (p / 'pyhydra').exists()),
    _cwd,
)
DATA_DIR = Path(os.environ.get('HYDRA_DATA_DIR', str(REPO_ROOT / 'data')))

RUN_DOWNLOADS = os.getenv('HYDRA_RUN_DOWNLOADS', '0') == '1'
OUTPUT_DIR = Path(os.getenv('HYDRA_RUNTIME_DIR', str(Path.cwd() / '.hydra_runtime'))) / 'persiann'
print(f'Run downloads: {RUN_DOWNLOADS} | Output dir: {OUTPUT_DIR}')


Run downloads: False | Output dir: /Users/salvadornavasfernandez/Desktop/Github/HYDRA/notebooks/data_sources/rainfall/.hydra_runtime/persiann


## 🗺️ Select area / point of interest

Draw a **rectangle** for gridded download or click a **point** for a single location.

In [2]:
coordinates_list = interactive_map(zoom=4, center=(20, 0))

In [3]:
# Define spatial domain — from the map above or hardcoded:
# For a point:
lat = 39.392231
lon = -0.624717

# For a bounding box:
# lon_min = coordinates_list[0][1]
# lon_max = coordinates_list[0][3]
# lat_min = coordinates_list[0][2]
# lat_max = coordinates_list[0][0]
lon_min = -3.251953
lon_max =  1.40625
lat_min = 37.195331
lat_max = 40.446947

---
## 🎯 Point download vs gridded download

PERSIANN-CCS can be downloaded in two modes:

| Mode | Use case | Output |
|------|----------|--------|
| **Point** (`lat`, `lon`) | Single gauge location, time series analysis | `pandas.Series` (mm/hr) |
| **Gridded** (bounding box) | Spatial fields, catchment-average, maps | NetCDF file |

#### Converting hourly → daily

```python
daily_mm = series.resample('D').sum()  # sum of 24 hourly values (each in mm/hr × 1hr = mm)
```

#### Estimating catchment-average precipitation

```python
import xarray as xr
ds = xr.open_dataset('persiann_area/grid.nc')
catchment_mean = ds['prcp'].mean(dim=['lat', 'lon'])  # simple spatial mean
# For weighted mean by cell area: use ds['prcp'].weighted(np.cos(np.deg2rad(ds.lat))).mean()
```


In [4]:
if RUN_DOWNLOADS:
    # --- Point download ---
    processor = PERSSIANDownloader(
        lat=lat, lon=lon,
        path_output=str(OUTPUT_DIR / 'point'))
else:
    print('PERSIANN operation skipped in public mode. Set HYDRA_RUN_DOWNLOADS=1 to run it.')


PERSIANN operation skipped in public mode. Set HYDRA_RUN_DOWNLOADS=1 to run it.


In [5]:
if RUN_DOWNLOADS:
    series = processor.download_hourly(
        start_date='2024-10-29',
        end_date='2024-10-30')
else:
    print('PERSIANN operation skipped in public mode. Set HYDRA_RUN_DOWNLOADS=1 to run it.')


PERSIANN operation skipped in public mode. Set HYDRA_RUN_DOWNLOADS=1 to run it.


In [6]:
if RUN_DOWNLOADS:
    series.plot()

In [7]:
if RUN_DOWNLOADS:
    # --- Gridded download ---
    processor = PERSSIANDownloader(
        lon_min=lon_min, lon_max=lon_max, lat_min=lat_min, lat_max=lat_max,
        path_output=str(OUTPUT_DIR / 'grid'))
else:
    print('PERSIANN operation skipped in public mode. Set HYDRA_RUN_DOWNLOADS=1 to run it.')


PERSIANN operation skipped in public mode. Set HYDRA_RUN_DOWNLOADS=1 to run it.


In [8]:
if RUN_DOWNLOADS:
    series = processor.download_hourly(
        start_date='2024-10-29',
        end_date='2024-10-30')
else:
    print('PERSIANN operation skipped in public mode. Set HYDRA_RUN_DOWNLOADS=1 to run it.')


PERSIANN operation skipped in public mode. Set HYDRA_RUN_DOWNLOADS=1 to run it.


In [9]:
if RUN_DOWNLOADS:
    try:
        processor.create_animation(
            str(OUTPUT_DIR / 'grid'), '2024-10-29', '2024-10-30', '1h', 'malla.gif'
        )
    except AttributeError:
        print('create_animation not yet implemented in PERSSIANDownloader — skipped.')
else:
    print('PERSIANN operation skipped in public mode. Set HYDRA_RUN_DOWNLOADS=1 to run it.')


PERSIANN operation skipped in public mode. Set HYDRA_RUN_DOWNLOADS=1 to run it.


---
## 📊 Post-download validation and bias correction

Before using PERSIANN-CCS as hydrological model forcing, compare against ground gauges:

```python
# Monthly comparison (PERSIANN vs AEMET)
persiann_monthly = series.resample('ME').sum()
aemet_monthly = aemet_series.resample('ME').sum()

# Bias factor (multiplicative correction)
bias = aemet_monthly.mean() / persiann_monthly.mean()
persiann_corrected = series * bias
```

Typical bias factors for PERSIANN-CCS in Spain:
- Coast / low elevation: 0.9–1.2 (good agreement)
- Mountain / interior: 1.3–2.0 (significant underestimation)
- Convective events (DANA type): bias can exceed 3× for individual events

#### Data volume
- 1 year hourly at 4 km over Iberia (~500×600 km²): **~5 GB**
- Use `download_daily()` when sub-daily resolution is not required (reduces size 24×)
